# 03 · Model Experiments & Spatial Validation

Evaluating Random Forest, Ridge, and Gradient Boosting baselines under random vs. spatial cross-validation schemes.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import KFold, GroupKFold

from isro_aqi.synthetic import SyntheticConfig, generate_all
from isro_aqi.preprocessing.collocate import sample_at_stations, join_targets
from isro_aqi.features import add_engineered_features


### 1. Build Training Dataset
Prepare engineered features (cyclical time, boundary layer ratios, spatial coordinates).


In [ ]:
cfg = SyntheticConfig(resolution_deg=0.5, n_days=30, n_stations=60)
data = generate_all(cfg)

df = join_targets(sample_at_stations(data["stack"], data["stations"]), data["observations"])
df = add_engineered_features(df)

feature_cols = [c for c in df.columns if c not in ["station_id", "date", "pm25", "pm10", "no2_obs", "so2_obs", "o3_obs", "co_obs"]]
X = df[feature_cols].fillna(0)
y = df["pm25"]
print(f"Features ({len(feature_cols)}):", feature_cols[:8], "...")


### 2. Random CV vs. Spatial GroupKFold CV
Demonstrate how spatial autocorrelation can lead to overoptimistic performance in random splits vs held-out geographic stations.


In [ ]:
# Random CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)
r2_random = []
for train_idx, val_idx in kf.split(X):
    rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
    rf.fit(X.iloc[train_idx], y.iloc[train_idx])
    preds = rf.predict(X.iloc[val_idx])
    r2_random.append(r2_score(y.iloc[val_idx], preds))

# Spatial GroupKFold CV (by station_id)
gkf = GroupKFold(n_splits=5)
r2_spatial = []
for train_idx, val_idx in gkf.split(X, groups=df["station_id"]):
    rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
    rf.fit(X.iloc[train_idx], y.iloc[train_idx])
    preds = rf.predict(X.iloc[val_idx])
    r2_spatial.append(r2_score(y.iloc[val_idx], preds))

print(f"Random CV R²:  {np.mean(r2_random):.3f} ± {np.std(r2_random):.3f}")
print(f"Spatial CV R²: {np.mean(r2_spatial):.3f} ± {np.std(r2_spatial):.3f}")
